# Train Lora + LLM Judge + RAG Evaluation

**Nguyen tac cua notebook nay:**
- **1 co `USE_RAG` DUY NHAT** dung chung cho Buoc 5 (run_evaluation) va
  Buoc 6 (LLM Judge) -- de dam bao evaluate LUON khop dieu kien luc train
  (train khong RAG thi evaluate cung khong RAG, va nguoc lai).
- **Chat (Buoc 7)** la doc lap -- tu chon bat/tat RAG rieng cho tung cau
  hoi bang lenh `/rag on` / `/rag off`, KHONG anh huong den co USE_RAG
  chung o tren.

**QUAN TRONG -- doc truoc khi chay:**
1. Runtime -> Disconnect and delete runtime (don sach session cu).
2. Runtime -> Change runtime type -> GPU (T4).
3. Chay tuan tu tu tren xuong, KHONG bo qua cell nao.
4. Sau Buoc 3 (cai dat), co cell yeu cau RESTART SESSION -- bat buoc.
5. Buoc 5 (diagnostic) neu co dong "[FAIL]" thi DUNG LAI xu ly truoc.


# Phần 1: Chuẩn bị cài đặt môi trường

## Buoc 0: Kiem tra GPU

In [1]:
!nvidia-smi


Sun Jul 19 07:56:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n[FAIL] KHONG CO GPU. Doi Runtime type -> GPU (T4), roi Disconnect and delete runtime.")
else:
    print("[OK] GPU san sang.")


torch.cuda.is_available(): True
[OK] GPU san sang.


## Buoc 1: Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Buoc 2: Clone project SACH + PATCH ngay (truoc khi cai dat)

Patch `trust_remote_code=True` cho nomic-embed-text phai lam NGAY SAU KHI
CLONE, truoc ca buoc cai dat/diagnostic -- neu de sau se bi FAIL o buoc
kiem tra rag_bridge.

In [4]:
import shutil, os

PROJECT_DIR = "/content/MockProject_062026_NhomAI"

if os.path.exists(PROJECT_DIR):
    print(f"Xoa ban clone cu tai {PROJECT_DIR} ...")
    shutil.rmtree(PROJECT_DIR)

for stray in ["/content/MockProject_062026_NhomAI"]:
    if os.path.exists(stray):
        print(f"Xoa ban clone LAC tai {stray} ...")
        shutil.rmtree(stray)

print("Da don sach. Bat dau clone moi...")


Da don sach. Bat dau clone moi...


In [5]:
!git clone --branch tuanphat --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git /content/MockProject_062026_NhomAI

%cd /content/MockProject_062026_NhomAI/Training

!git clone --branch han --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git _RAG_


Cloning into '/content/MockProject_062026_NhomAI'...
remote: Enumerating objects: 11631, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 11631 (delta 29), reused 25 (delta 19), pack-reused 11579 (from 2)
Receiving objects: 100% (11631/11631), 176.12 MiB | 19.74 MiB/s, done.
Resolving deltas: 100% (6969/6969), done.
Updating files: 100% (11422/11422), done.
/content/MockProject_062026_NhomAI/Training
Cloning into '_RAG_'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 222 (delta 3), reused 2 (delta 2), pack-reused 206 (from 1)
Receiving objects: 100% (222/222), 177.11 MiB | 18.66 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (116/116), done.


In [6]:
import os
assert os.getcwd() == "/content/MockProject_062026_NhomAI/Training", f"Sai working dir: {os.getcwd()}"
assert os.path.exists("pipeline/evaluate.py"), "Khong tim thay pipeline/evaluate.py -- clone loi"
assert os.path.exists("_RAG_/Embbeding_RAG"), "Khong tim thay _RAG_/Embbeding_RAG -- clone loi"
print("[OK] Clone dung vi tri, cau truc thu muc hop le.")


[OK] Clone dung vi tri, cau truc thu muc hop le.


In [7]:
# PATCH: nomic-embed-text-v1.5 can trust_remote_code=True (kien truc custom)
NOMIC_DIR = "/content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5"

!sed -i 's/model_name=embedding_model_name$/model_name=embedding_model_name, trust_remote_code=True/' \
  "{NOMIC_DIR}/test_vector_db/retrieval.py"
!sed -i 's/model_name=EMBEDDING_MODEL_NAME$/model_name=EMBEDDING_MODEL_NAME, trust_remote_code=True/' \
  "{NOMIC_DIR}/embedding_storage.py"

# Xac nhan patch da vao dung file
!grep -n "trust_remote_code" "{NOMIC_DIR}/test_vector_db/retrieval.py" || echo "[FAIL] Patch chua vao retrieval.py"


139:            model_name=embedding_model_name, trust_remote_code=True


## Buoc 3: Cai dat package (1 LAN DUY NHAT, du het trong 1 lenh)

In [8]:
%cd /content/MockProject_062026_NhomAI/Training

!pip install --upgrade pip -q
!pip install \
    -r requirements_ver1.txt \
    -r _RAG_/Embbeding_RAG/requirements.txt \
    -q
!pip install -U "opentelemetry-api==1.44.0" "opentelemetry-sdk==1.44.0" -q


/content/MockProject_062026_NhomAI/Training
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.38 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you hav

## >>> BAT BUOC: RESTART SESSION NGAY BAY GIO <<<

Runtime -> Restart session. Sau khi restart, chay tiep tu Buoc 4 -- KHONG
chay lai Buoc 0-3.

In [2]:
print("Nho: Runtime -> Restart session, roi moi chay tiep Buoc 4.")


Nho: Runtime -> Restart session, roi moi chay tiep Buoc 4.


## Buoc 4: DAT CO USE_RAG CHUNG (dung cho ca Buoc 5 va Buoc 6)

Doi gia tri True/False O DAY DUY NHAT -- dam bao evaluate luon khop dieu
kien luc train (vd: train KHONG dung RAG -> de False; neu ban co train
lai VOI RAG thi doi thanh True).

In [3]:
import os
os.chdir("/content/MockProject_062026_NhomAI/Training")

import sys
sys.path.insert(0, "/content/MockProject_062026_NhomAI/Training")

# ============================================================
# >>> DOI GIA TRI NAY CHO KHOP VOI LUC TRAIN <<<
TRAIN_EVAL_USE_RAG = False   # True neu model duoc train/muon eval CO RAG
# ============================================================

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MEDQUAD_RAG_DIR"] = "/content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5"
os.environ["MEDQUAD_USE_RAG"] = "1" if TRAIN_EVAL_USE_RAG else "0"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

print(f"[OK] Da set MEDQUAD_USE_RAG = {os.environ['MEDQUAD_USE_RAG']} (dung cho Buoc 5 + Buoc 6)")


[OK] Da set MEDQUAD_USE_RAG = 0 (dung cho Buoc 5 + Buoc 6)


## Buoc 5: DIAGNOSTIC -- kiem tra du dieu kien truoc khi chay that

In [4]:
import torch
print(("[OK]" if torch.cuda.is_available() else "[FAIL]"), "GPU:", torch.cuda.is_available())


[OK] GPU: True


In [5]:
try:
    import opentelemetry.sdk.environment_variables as ev
    _ = ev.OTEL_LOGRECORD_ATTRIBUTE_COUNT_LIMIT
    print("[OK] opentelemetry OK. File:", ev.__file__)
except Exception as e:
    print("[FAIL] opentelemetry loi:", e)


[OK] opentelemetry OK. File: /usr/local/lib/python3.12/dist-packages/opentelemetry/sdk/environment_variables/__init__.py


In [6]:
try:
    import chromadb
    print("[OK] chromadb OK, version:", chromadb.__version__)
except Exception as e:
    print("[FAIL] chromadb loi:", e)


[OK] chromadb OK, version: 1.5.9


In [7]:
try:
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    print("[OK] ragas OK.")
except Exception as e:
    print("[FAIL] ragas loi:", e)


[OK] ragas OK.


In [8]:
try:
    from src.config import USE_RAG, ADAPTER_DIR, PREDICTIONS_CSV
    print("[OK] src.config OK. USE_RAG =", USE_RAG)
    print("ADAPTER_DIR:", ADAPTER_DIR, "| ton tai:", ADAPTER_DIR.exists())
    print("PREDICTIONS_CSV:", PREDICTIONS_CSV)
except Exception as e:
    print("[FAIL] src.config loi:", e)


[OK] src.config OK. USE_RAG = False
ADAPTER_DIR: /content/MockProject_062026_NhomAI/Training/output/output_model | ton tai: True
PREDICTIONS_CSV: /content/MockProject_062026_NhomAI/Training/output/evaluation_results.csv


In [9]:
from src.config import USE_RAG
import os

if USE_RAG:
    rag_dir = os.environ["MEDQUAD_RAG_DIR"]
    chroma_path = os.path.join(rag_dir, "chroma_db", "chroma.sqlite3")
    if os.path.exists(chroma_path):
        print("[OK] chroma_db ton tai tai:", chroma_path)
    else:
        print(f"[FAIL] KHONG tim thay {chroma_path} -- can build chroma_db truoc.")
else:
    print("[SKIP] USE_RAG=False -- khong can check chroma_db.")


[SKIP] USE_RAG=False -- khong can check chroma_db.


In [10]:
from src.config import USE_RAG

if USE_RAG:
    try:
        from src.rag_bridge import get_context_with_similarity
        r = get_context_with_similarity("What are the symptoms of diabetes?", top_k=2)
        if r["raw_contexts"]:
            print(f"[OK] rag_bridge retrieve OK -- {len(r['raw_contexts'])} chunks, rag_used={r['rag_used']}")
            print("Preview:", r["raw_contexts"][0][:200])
        else:
            print("[FAIL] rag_bridge chay khong loi nhung RONG -- kiem tra lai chroma_db co du lieu chua.")
    except Exception as e:
        print("[FAIL] rag_bridge loi:", type(e).__name__, "-", e)
else:
    print("[SKIP] USE_RAG=False -- khong can test rag_bridge.")


[SKIP] USE_RAG=False -- khong can test rag_bridge.


In [11]:
from src.config import ADAPTER_DIR
if ADAPTER_DIR.exists() and any(ADAPTER_DIR.iterdir()):
    print("[OK] Tim thay model da train tai:", ADAPTER_DIR)
else:
    print(f"[FAIL] Chua co model tai {ADAPTER_DIR} -- can train truoc hoac tai model len dung vi tri nay.")


[OK] Tim thay model da train tai: /content/MockProject_062026_NhomAI/Training/output/output_model


In [12]:
print("=" * 60)
print("Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.")
print("=" * 60)


Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.


# Phần 2: Train

## Bước 1: Build train.jsonl từ final_train_dataset.json

In [13]:
from pipeline import build_train_dataset
build_train_dataset.main()

Total samples (raw)   : 37172
Sample limit áp dụng  : None
Hợp lệ (validated)    : 37172
Bỏ qua (thiếu Q/A)    : 0
USE_RAG               : False
----------------------------------------
Train : 26020 -> /content/MockProject_062026_NhomAI/Training/output/train.jsonl
Val   : 5575 -> /content/MockProject_062026_NhomAI/Training/output/val.jsonl
Test  : 5577 -> /content/MockProject_062026_NhomAI/Training/output/test.jsonl


## Bước 2: Train Lora

### Bước 2.1: Training (Nếu đã có model -> Bỏ qua)

In [14]:
from pipeline import train
train.main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model + tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Đang chạy sanity check (gradient có lan tới LoRA không)...
  Loss thử (1 mẫu)        : 3.9937
  LoRA params có gradient : 96/192
  CẢNH BÁO: chỉ 96/192 LoRA params có gradient (không phải tất cả). Có thể vẫn train được nhưng nên kiểm tra lại target_modules nếu kết quả cuối không như mong đợi.
  -> OK, gradient lan tới LoRA bình thường. Bắt đầu train thật.

Bắt đầu train...
Eval mỗi 200 step trên val set -- sẽ tự dừng sớm nếu eval_loss không cải thiện sau 3 lần eval liên tiếp, và tự chọn checkpoint có eval_loss thấp nhất (điểm tối ưu) khi xong.


  0%|          | 0/3253 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss


{'loss': 2.4483, 'grad_norm': 1.712815284729004, 'learning_rate': 0.00019944666461727638, 'num_tokens': 16985.0, 'mean_token_accuracy': 0.5182959869503975, 'epoch': 0.0}


KeyboardInterrupt: 

### Bước 2.2: Nếu đã có model

In [15]:
from pipeline import run_evaluation
run_evaluation.main()   # full tap test, ghi đè từ đầu
# run_evaluation.main(resume=True)  # dùng nếu bị ngắt giữa chừng


Đang tải tập test (thô)...
Đang load model đã train từ /content/MockProject_062026_NhomAI/Training/output/output_model...
Load xong.
Số câu hỏi test: 5577 | Sẽ chạy: 5577
USE_RAG=False -> chạy Q&A thuần, không có ngữ cảnh (như cũ).
[EVAL] Bắt đầu MỚI -- sẽ ghi đè /content/MockProject_062026_NhomAI/Training/output/evaluation_results.csv nếu đã tồn tại.
[EVAL] Mục tiêu: 5577 mẫu.
[EVAL] Đã inference 5/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 10/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 15/5577 mẫu... (đã lưu CSV)


KeyboardInterrupt: 

## Buoc 3: LLM Judge (RAGAs) -- van dung DUNG USE_RAG cua Buoc 4, khop voi Buoc 6

In [16]:
import getpass
os.environ["MEDQUAD_JUDGE_API_KEY"] = getpass.getpass("Nhập API Key")
os.environ["MEDQUAD_JUDGE_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["MEDQUAD_JUDGE_API_MODEL"] = "llama-3.1-8b-instant"

import importlib
import src.config
importlib.reload(src.config)   # doc lai JUDGE_API_KEY vua nhap

from pipeline import evaluate
importlib.reload(evaluate)
evaluate.main()


Nhập API Key··········
Kết quả đánh giá sẽ được lưu (append theo batch) vào: /content/drive/MyDrive/medquad_eval/ragas_scores.csv
Kết quả đánh giá sẽ được lưu (append theo batch) vào: /content/drive/MyDrive/medquad_eval/ragas_scores.csv
Đang đọc CSV dự đoán (đã sinh sẵn từ bước train, gồm ROUGE/BLEU)...
Số mẫu: 15
Đang khởi tạo model giám khảo (tách biệt model vừa train)...
Bật JSON mode cho giám khảo (giảm lỗi parse với model nhỏ). Nếu thấy TOÀN BỘ câu bị lỗi/NaN sau khi bật, set MEDQUAD_JUDGE_JSON_MODE=0 và chạy lại để tắt JSON mode.
Gọi model giám khảo qua API: llama-3.1-8b-instant (https://api.groq.com/openai/v1)


/content/MockProject_062026_NhomAI/Training/pipeline/evaluate.py:148: LangChainBetaWarning: Introduced in 0.2.24. API subject to change.
  rate_limiter = InMemoryRateLimiter(


Đang load embedding model (cho Answer Relevance)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Đã lưu batch 1 vào /content/drive/MyDrive/medquad_eval/ragas_scores.csv
--- Batch 2 (5 câu, 10/15) ---


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Đã lưu batch 2 vào /content/drive/MyDrive/medquad_eval/ragas_scores.csv
--- Batch 3 (5 câu, 15/15) ---


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Đã lưu batch 3 vào /content/drive/MyDrive/medquad_eval/ragas_scores.csv
KẾT QUẢ ĐÁNH GIÁ (RAGAs + LLM Judge)
Tổng số câu đã chấm: 20
answer_relevancy    0.770633
dtype: float64

Chi tiết đầy đủ: /content/drive/MyDrive/medquad_eval/ragas_scores.csv


## Buoc 4: CHAT -- doc lap, tu chon bat/tat RAG rieng (khong lien quan co USE_RAG o Buoc 4)

Trong luc chat, go:
- `/rag on`  -> bat RAG cho cac cau hoi tiep theo
- `/rag off` -> tat RAG (Q&A thuan)
- `exit` hoac `quit` -> thoat

Neu USE_RAG cua ban chua bat (Buoc 4 dang False) ma muon thu RAG o day,
can dam bao da co chroma_db + da patch trust_remote_code (Buoc 2).

In [ ]:
from pipeline import chat
chat.chat_loop()


Đang load model...
Load model xong.

CHATBOT SẴN SÀNG (mặc định USE_RAG=False)
Lệnh: '/rag on' bật RAG | '/rag off' tắt RAG | 'exit'/'quit' thoát

Câu hỏi của bạn: /rag on
[OK] Đã BẬT RAG cho các câu hỏi tiếp theo.

Câu hỏi của bạn: /rag off
[OK] Đã TẮT RAG cho các câu hỏi tiếp theo (Q&A thuần).

Câu hỏi của bạn: How to take care of a patient with heart disease

TRẢ LỜI: Follow your doctor's instructions for medication use and follow-up appointments.

Câu hỏi của bạn: /rag on
[OK] Đã BẬT RAG cho các câu hỏi tiếp theo.

Câu hỏi của bạn: How to take care of a patient with heart disease
[rag_bridge] Đã load retrieval module từ: /content/test_llm_project/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5
[rag_bridge] RETRIEVAL_MODE = hybrid | SIMILARITY_THRESHOLD = 0.7


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

[rag_bridge][CẢNH BÁO] RETRIEVAL_MODE='hybrid' -- score không quy đổi được thành % tương đồng chuẩn, nên KHÔNG lọc theo SIMILARITY_THRESHOLD. Toàn bộ context retrieve được sẽ được dùng. Nếu muốn lọc theo threshold đúng nghĩa, đổi RETRIEVAL_MODE='cosine' trong config.py của RAG project.

--- RAG cho câu hỏi này (mode=hybrid) ---
  [1] Tương đồng: N/A (mode không quy đổi được % tương đồng) | Rheumatic heart disease is inflammatory damage of the heart valves, as a complication of acute rheumatic fever. The mitral valve is the most commonly ...
  [2] Tương đồng: N/A (mode không quy đổi được % tương đồng) | 1. Resident T has a diagnosis of heart failure. During the past provider. Under the hospice few months, they have had three hospital admissions for pr...
  [3] Tương đồng: N/A (mode không quy đổi được % tương đồng) | □ • Treatable medical conditions, such as heart...
  -> DÙNG RAG: 3 context đạt ngưỡng, đưa vào prompt.
--------------------------------------------------

TRẢ LỜI: Taking s